In [1]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

In [2]:
from langchain.embeddings import init_embeddings
from langgraph.store.memory import InMemoryStore
import uuid
from datetime import datetime

embeddings = init_embeddings("openai:text-embedding-3-small")
store = InMemoryStore(
    index={
        "embed": embeddings,
        "dims": 1536,
    }
)

store.put(
    ("user",),
    "user_123",
    {
        "user_name": "mina",
        "user_age": 25,
    }
)

In [3]:
user_info = store.get(("user",), "user_123")
user_info

Item(namespace=['user'], key='user_123', value={'user_name': 'mina', 'user_age': 25}, created_at='2026-09-20T05:25:44.488087+00:00', updated_at='2026-09-20T05:25:44.488091+00:00')

In [4]:
from langchain_core.runnables import RunnableConfig
from langgraph.config import get_store
from langchain.tools import tool

@tool
def get_user_info(config: RunnableConfig) -> str:
    """사용자의 기본 정보를 조회합니다."""
    store = get_store()
    user_id = config["configurable"].get("user_id")

    user_info = store.get(("user",), user_id)
    return str(user_info.value) if user_info else "알 수 없는 사용자"

In [5]:
@tool
def save_user_info(
    preferences: list[str] = None,
    interests: list[str] = None,
    experiences: list[str] = None,
    current_activities: list[str] = None,
    goals: list[str] = None,
    routines: list[str] = None,
    concerns: list[str] = None,
    achievements: list[str] = None,
    config: RunnableConfig = None,
) -> str:
    """사용자의 다양한 정보를 카테고리별 컬렉션에 저장합니다.
    
    Args:
        preferences: 사용자가 선호하는 것들 (활동, 스타일, 방식 등)
        interests: 사용자가 관심있어 하는 주제나 분야
        experiences: 사용자가 과거에 경험한 것들
        current_activities: 사용자가 현재 진행 중인 활동이나 프로젝트
        goals: 사용자의 목표 (단기/장기)
        routines: 사용자의 일상 루틴이나 습관
        concerns: 사용자의 현재 고민이나 문제
        achievements: 사용자의 성취나 긍정적 피드백
        config: user_id가 포함된 RunnableConfig

    각 정보는 벡터 검색이 가능하도록 카테고리별 컬렉션에 개별 아이템으로 저장됩니다.
    """

    store = get_store()
    user_id = config["configurable"].get("user_id")
    current_time = datetime.now().isoformat()

    categories = {
        "preferences": preferences,
        "interests": interests,
        "experiences": experiences,
        "current_activities": current_activities,
        "goals": goals,
        "routines": routines,
        "concerns": concerns,
        "achievements": achievements
    }

    update_summary = []
    for category, values in categories.items():
        if values:
            for value in values:
                item_id = str(uuid.uuid4())
                store.put(
                    (user_id, category),
                    item_id,
                    {
                        "text": value,
                        "created_at": current_time,
                        "category": category,
                    }
                )
            update_summary.append(f"{len(values)}개의 {category}")
            
    return f"사용자 메모리가 성공적으로 저장되었습니다: {', '.join(update_summary)}가 추가되었습니다."

In [6]:
@tool
def search_user_memories(
    query: str,
    category: str = None,
    limit: int = 5,
    config: RunnableConfig = None,
) -> str:
    """사용자의 메모리를 검색합니다. 특정 상황이나 질문과 관련된 과거 정보를 찾을 때 사용합니다.
    
    Args:
        query: 검색할 내용
        category: 특정 카테고리로 제한 (preferences, interests, experiences, current_activities, goals, routines, concerns, achievements)
        limit: 반환할 최대 결과 수
        config: user_id가 포함된 RunnableConfig
    """

    store = get_store()
    user_id = config["configurable"].get("user_id")

    if category:
        namespace = (user_id, category)
        results = store.search(namespace, query=query, limit=limit)
        if not results:
            return f"{category} 카테고리에서 관련된 메모리를 찾을 수 없습니다."

        result_text = f"{category} 관련 메모리:\n"
        for item in results:
            result_text += f"- {item.value['text']} (저장일자: {item.value['created_at']})\n"
        return result_text

    else:
        categories = ["preferences", "interests", "experiences", "current_activities", "goals", "routines", "concerns", "achievements"]
        all_results = []
        for cat in categories:
            try:
                results = store.search((user_id, cat), query=query, limit=limit)
                all_results.extend([(r, cat) for r in results])
            except:
                continue

        all_results.sort(key=lambda x: x[0].score if hasattr(x[0], 'score') else 0, reverse=True)
        all_results = all_results[:limit]

        if not all_results:
            return "관련된 메모리를 찾을 수 없습니다."

        result_text = "관련 메모리:\n"
        for item, cat in all_results:
            result_text += f"[{cat}] {item.value['text']} (저장일자: {item.value['created_at']})\n"
        return result_text


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-4o")
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools = [get_user_info, save_user_info, search_user_memories],
    store=store,
    system_prompt= """
        당신은 누적된 사용자 메모리를 활용하여 맞춤 조언을 제공하는 친절한 라이프 코치 어시스턴트입니다.

        **도구 활요 가이드:**

        1. **인사/대화 시작 시:**
            - get_user_info로 기본 정보 확인
            - search_user_memories로 current_activities 검색하여 "~는 잘되고 있나요?" 안부 묻기
        
        2. **조언 제공 시**
            - search_user_memories를 적극 활용해 관련 과거 정보 검색
            - 예: 운동 질문 -> query="운동", 고민 상담 -> concerns/routines 등 필요한 카테고리 선택하여 검색

        3. **정보 저장 시 (save_user_info)**
            사용자가 **명시적으로 자신의 정보를 밝힐 때만** 저장하세요.

            저장하는 경우:
            - preferences: "나는 아침형 인간이야", "조용한 카페를 좋아해"
            - interests: "요즘 AI 공부에 관심이 많아", "사진 찍는 게 취미야"
            - experiences: "작년에 등산 동아리 했었어"
            - current_activities: "요즘 다이어트 중이야", "파이썬 공부하고 있어"
            - goals: "다음 달까지 5kg 감량이 목표야"
            - routines: "주 3회 운동하고 있어", "매일 명상 10분씩 해"
            - concerns: "요즘 집중력이 떨어져", "업무 스트레스가 심해"
            - achievements: "오늘 10km 달리기 성공했어", "프로젝트 마감 잘 끝냈어"

        **응답 가이드:**
        - 개인 정보(나이, 사용자 ID) 직접 언급 금지
        - 누적 정보 활용: 목표-루틴 연결, 성취 격려, 고민 해결
        - 자연스럽고 공감하며 실천 가능한 조언 제공
    """,
    checkpointer=checkpointer,
)

In [8]:
while True:
    user_input = input("User: ")
    if user_input.lower() in ["q", "exit", "quit"]:
        break

    response = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config={"configurable": {"user_id": "user_123", "thread_id": "1"}}
    )

    for msg in response["messages"]:
        msg.pretty_print()

================================ Human Message =================================

안녕! 나 요즘 다이어트 중이라, 주 3회씩 운동하고 있는데 힘드네 ㅠㅠ 얼른 5kg 정도 감량하고 싶어.
================================== Ai Message ==================================
Tool Calls:
  save_user_info (call_VLrzOYdUiQJOd18kW9UmMPhD)
 Call ID: call_VLrzOYdUiQJOd18kW9UmMPhD
  Args:
    current_activities: ['다이어트 중']
    goals: ['5kg 감량']
    routines: ['주 3회 운동하고 있어']
  search_user_memories (call_ufOhTxit2W3hrRn5MmNmBa5d)
 Call ID: call_ufOhTxit2W3hrRn5MmNmBa5d
  Args:
    query: 운동
    category: current_activities
================================= Tool Message =================================
Name: save_user_info

사용자 메모리가 성공적으로 저장되었습니다: 1개의 current_activities, 1개의 goals, 1개의 routines가 추가되었습니다.
================================= Tool Message =================================
Name: search_user_memories

current_activities 카테고리에서 관련된 메모리를 찾을 수 없습니다.
================================== Ai Message ==================================

안녕! 다이어트